# genQC Paper SRV Evaluation

Compact notebook to:
1. reproduce the SRV benchmark used in the genQC paper, and
2. compare multiple trained models on exactly those same metrics.

This notebook is intentionally limited to the SRV task. It uses the existing `SRVEvaluator` for sampling, decoding, and SRV extraction, then computes the paper-style macro accuracy by number of entangled qubits.


## 1. Setup


In [ ]:
import os
import sys
import random
from collections import defaultdict
from pathlib import Path

import hydra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from quantum_diffusion.evaluation.evaluator import SRVEvaluator

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 200)


def as_path(path_like: str | Path) -> Path:
    return Path(path_like).expanduser().resolve()


In [ ]:
# Edit only this cell.
DATASET_PATH = "./datasets/paper_quditkit/srv_8q_dataset"
NUM_SAMPLES = 5000
SEED = 1234

MODEL_SPECS = [
    {
        "label": "paper_srv_model",
        "model_dir": "./models/trained/paper_srv_model",
    },
]


## 2. Helpers


In [ ]:
def build_eval_config(dataset_path: str | Path, model_dir: str | Path, num_samples: int):
    with hydra.initialize(version_base=None, config_path="conf"):
        cfg = hydra.compose(config_name="config.yaml", overrides=["evaluation=paper_srv"])
    cfg = cfg["evaluation"]
    cfg.dataset = str(as_path(dataset_path))
    cfg.model_dir = str(as_path(model_dir))
    cfg.hf_repo = None
    cfg.num_samples = int(num_samples)
    cfg.save_output = False
    cfg.wandb.enable = False
    return cfg


def paper_macro_accuracy_by_entangled_qubits(target_srvs: torch.Tensor, predicted_srvs: torch.Tensor) -> dict[int, float]:
    if target_srvs.numel() == 0:
        return {}

    total_per_srv = defaultdict(int)
    correct_per_srv = defaultdict(int)

    for target, predicted in zip(target_srvs.cpu().tolist(), predicted_srvs.cpu().tolist()):
        srv_key = tuple(int(v) for v in target)
        total_per_srv[srv_key] += 1
        if tuple(int(v) for v in predicted) == srv_key:
            correct_per_srv[srv_key] += 1

    grouped = defaultdict(list)
    for srv_key, total in total_per_srv.items():
        n_entangled = sum(1 for value in srv_key if value == 2)
        grouped[n_entangled].append(correct_per_srv[srv_key] / total)

    return {bucket: float(np.mean(values)) for bucket, values in sorted(grouped.items())}


def micro_accuracy_by_entangled_qubits(target_srvs: torch.Tensor, predicted_srvs: torch.Tensor, num_qubits: int) -> dict[int, float]:
    if target_srvs.numel() == 0:
        return {}

    counts = {n: 0 for n in range(num_qubits + 1) if n != 1}
    correct = {n: 0 for n in counts}

    for target, predicted in zip(target_srvs, predicted_srvs):
        bucket = int((target == 2).sum().item())
        counts[bucket] += 1
        if torch.equal(target, predicted):
            correct[bucket] += 1

    return {bucket: correct[bucket] / counts[bucket] for bucket in counts if counts[bucket] > 0}


def compact_bucket_string(bucket_metrics: dict[int, float]) -> str:
    if not bucket_metrics:
        return "-"
    return " | ".join(f"{bucket}:{value:.3f}" for bucket, value in sorted(bucket_metrics.items()))


def evaluate_model(model_spec: dict, dataset_path: str | Path, num_samples: int, seed: int) -> dict:
    cfg = build_eval_config(dataset_path=dataset_path, model_dir=model_spec["model_dir"], num_samples=num_samples)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    evaluator = SRVEvaluator(config=cfg)
    tensors_out = evaluator.generate_tensors(save_output=False)
    decoded_circuits = evaluator.decode_tensors(tensors_out)
    valid_indices, target_srvs, predicted_srvs = evaluator.validate_and_calculate_srvs(decoded_circuits, save_output=False)

    valid_decodes = len(valid_indices)
    requested = evaluator.samples
    conversion_rate = valid_decodes / requested if requested else 0.0

    if valid_decodes > 0:
        srv_exact_match_rate = evaluator._get_exact_match_rate(target_srvs, predicted_srvs)
        paper_macro = paper_macro_accuracy_by_entangled_qubits(target_srvs, predicted_srvs)
        micro_acc = micro_accuracy_by_entangled_qubits(target_srvs, predicted_srvs, evaluator.num_qubits)
        macro_mean = float(np.mean(list(paper_macro.values()))) if paper_macro else float("nan")
    else:
        srv_exact_match_rate = 0.0
        paper_macro = {}
        micro_acc = {}
        macro_mean = float("nan")

    return {
        "label": model_spec["label"],
        "model_dir": str(as_path(model_spec["model_dir"])),
        "dataset_path": str(as_path(dataset_path)),
        "requested_samples": requested,
        "valid_decodes": valid_decodes,
        "decode_failures": requested - valid_decodes,
        "conversion_rate": conversion_rate,
        "srv_exact_match_rate": srv_exact_match_rate,
        "paper_macro_accuracy_by_entangled_qubits": paper_macro,
        "micro_accuracy_by_entangled_qubits": micro_acc,
        "paper_macro_accuracy_mean": macro_mean,
        "num_qubits": evaluator.num_qubits,
        "target_srvs": target_srvs.cpu(),
        "predicted_srvs": predicted_srvs.cpu(),
    }


def plot_paper_macro_accuracy(results_by_model: dict[str, dict]):
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=140)

    plotted = False
    for label, result in results_by_model.items():
        bucket_metrics = result["paper_macro_accuracy_by_entangled_qubits"]
        if not bucket_metrics:
            continue
        xs = sorted(bucket_metrics)
        ys = [bucket_metrics[x] for x in xs]
        ax.plot(xs, ys, marker="o", linewidth=1.8, markersize=4, label=label)
        plotted = True

    ax.set_xlabel("Entangled qubits")
    ax.set_ylabel("Macro exact-match accuracy")
    ax.set_title("genQC paper SRV metric")
    ax.set_ylim(0, 1.05)
    if plotted:
        ax.legend(title="Model")
    else:
        ax.text(0.5, 0.5, "No valid decodes available.", ha="center", va="center", transform=ax.transAxes)
    plt.tight_layout()
    plt.show()


def build_summary_table(results_by_model: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for label, result in results_by_model.items():
        rows.append({
            "model": label,
            "model_dir": result["model_dir"],
            "requested_samples": result["requested_samples"],
            "valid_decodes": result["valid_decodes"],
            "decode_failures": result["decode_failures"],
            "conversion_rate": result["conversion_rate"],
            "srv_exact_match_rate": result["srv_exact_match_rate"],
            "paper_macro_accuracy_mean": result["paper_macro_accuracy_mean"],
            "paper_macro_by_bucket": compact_bucket_string(result["paper_macro_accuracy_by_entangled_qubits"]),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    numeric_cols = [
        "conversion_rate",
        "srv_exact_match_rate",
        "paper_macro_accuracy_mean",
    ]
    df[numeric_cols] = df[numeric_cols].round(4)
    return df.sort_values(by=["srv_exact_match_rate", "conversion_rate"], ascending=False).reset_index(drop=True)


## 3. Evaluate Models


In [ ]:
results_by_model = {}

for offset, model_spec in enumerate(MODEL_SPECS):
    print(f"Evaluating {model_spec['label']} ...")
    results_by_model[model_spec["label"]] = evaluate_model(
        model_spec=model_spec,
        dataset_path=DATASET_PATH,
        num_samples=NUM_SAMPLES,
        seed=SEED + offset,
    )

print(f"Finished evaluating {len(results_by_model)} model(s).")


In [ ]:
summary_df = build_summary_table(results_by_model)
display(summary_df)


## 4. Paper Reproduction Plot


In [ ]:
plot_paper_macro_accuracy(results_by_model)


## 5. Optional Cross-Check: Current Evaluator Micro Accuracy


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=140)
plotted = False

for label, result in results_by_model.items():
    bucket_metrics = result["micro_accuracy_by_entangled_qubits"]
    if not bucket_metrics:
        continue
    xs = sorted(bucket_metrics)
    ys = [bucket_metrics[x] for x in xs]
    ax.plot(xs, ys, marker="s", linestyle="--", linewidth=1.4, markersize=4, label=label)
    plotted = True

ax.set_xlabel("Entangled qubits")
ax.set_ylabel("Micro exact-match accuracy")
ax.set_title("Current evaluator bucket accuracy")
ax.set_ylim(0, 1.05)
if plotted:
    ax.legend(title="Model")
else:
    ax.text(0.5, 0.5, "No valid decodes available.", ha="center", va="center", transform=ax.transAxes)
plt.tight_layout()
plt.show()


## Notes

- `conversion_rate` is `valid_decodes / requested_samples`.
- `srv_exact_match_rate` is computed on valid decodes only.
- The primary plot uses the paper-style macro averaging: exact-match accuracy is computed per distinct target SRV and then averaged within each entangled-qubit bucket.
- The optional micro plot shows the current evaluator behavior for comparison when the SRV frequency distribution is imbalanced.
